In [1]:
import requests
import time
import json
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame, functions as F, types as T
from delta.tables import DeltaTable


# Nome da tabela Delta
table_name = "mongodb_pedidos_products"

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
delta_table = DeltaTable.forName(spark, table_name)



# Função para desaninhamento completo (flatten)
def flatten_all(df: DataFrame, prefix: str = "") -> DataFrame:
    flat_cols = []

    for field in df.schema.fields:
        field_name = field.name
        col_name = f"{prefix}{field_name}"

        if isinstance(field.dataType, T.StructType):
            # Desaninha structs diretamente
            for subfield in field.dataType.fields:
                flat_cols.append(F.col(f"{field_name}.{subfield.name}").alias(f"{col_name}_{subfield.name}"))
        elif isinstance(field.dataType, T.ArrayType) and isinstance(field.dataType.elementType, T.StructType):
            # Explode arrays de structs
            df = df.withColumn(field_name, F.explode_outer(F.col(field_name)))
            return flatten_all(df, prefix=prefix)
        else:
            flat_cols.append(F.col(field_name).alias(col_name))

    df = df.select(flat_cols)

    # Reaplica se ainda houver structs
    if any(isinstance(field.dataType, T.StructType) for field in df.schema.fields):
        return flatten_all(df, prefix="")
    return df

# Passo 1: Descobrir o Total de Páginas
url_pages = (
    "https://apivesti.vesti.mobi/order/v1/orders/"
    "?filter[isClosed]=true"
    "&select=_id"
    "&filter[settings.createdAt][$gte]=2025-09-01"
    "&limit=600"
)

try:
    resposta = requests.get(url_pages)
    resposta.raise_for_status()
    mongo_json = resposta.json()
    totalPages = mongo_json.get('totalPages', 1)
    print(f'Total de páginas: {totalPages}')
except requests.exceptions.RequestException as e:
    print(f"Erro ao obter o total de páginas: {e}")
    totalPages = 0

# Passo 2: Coletar os Dados
endpoint_base = (
    "https://apivesti.vesti.mobi/order/v1/orders/"
    "?filter[isClosed]=true"
    "&filter[settings.createdAt][$gte]=2025-09-01"
    "&select=_id,products.id,products.integrationId,products.code,products.name,products.status,products.qnt,products.total,products.price,products.dimensions"
    "&limit=600"
# "&select=_id,companyId,domainId,settings.source,settings.createdAt,settings.affiliateId,"
# "orderNumber,payment.method,payment.type,payment.paidAt,payment.transaction.value,"
)

all_data = []
max_retries = 3
retry_delay = 5

if totalPages > 0:
    for page in range(1, totalPages + 1):
        print(f"Processando página {page}/{totalPages}...")
        retries = 0
        while retries < max_retries:
            try:
                response = requests.get(endpoint_base + f"&page={page}")
                response.raise_for_status()
                data = response.json()
                all_data.extend(data['data'])
                break
            except requests.exceptions.Timeout:
                retries += 1
                print(f"Tempo limite excedido na página {page}. Tentando novamente ({retries}/{max_retries})...")
                time.sleep(retry_delay)
            except requests.exceptions.RequestException as e:
                retries += 1
                print(f"Erro na requisição da página {page}: {e}. Tentando novamente ({retries}/{max_retries})...")
                time.sleep(retry_delay)
        else:
            print(f"Falha ao processar a página {page} após {max_retries} tentativas. Ignorando esta página.")

print(f"\nTotal de registros coletados: {len(all_data)}")

# Passo 3: Processar os dados e salvar na tabela Delta
if len(all_data) > 0:
    print("Criando Spark DataFrame com leitura via JSON...")
    rdd_json = spark.sparkContext.parallelize([json.dumps(item) for item in all_data])
    df_spark_initial = spark.read.json(rdd_json)

    # A função flatten_all transforma os dados para uma linha por produto
    df_flattened = flatten_all(df_spark_initial)
    print(f"\nNúmero de colunas após planificação: {len(df_flattened.columns)}")

    # # Salvar como Delta
    print(f"Salvando dados planificados na tabela Delta do Lakehouse: {table_name}")
    delta_table = DeltaTable.forName(spark, table_name)
    
    ## -----Essa forma apenas acrescenta os novos IDs
    # delta_table = DeltaTable.forName(spark, table_name)
    # delta_table.alias("tabela_existente").merge(
    #     df_flattened.alias("novos_dados"),
    #     "tabela_existente._id = novos_dados._id"  # substitua por sua chave única
    # ).whenNotMatchedInsertAll().execute()
    # print("Novos Pedidos Adicionados com sucesso.")

    ## ----- Essa forma Substitui toda a Tabela
    # df_flattened.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table_name)
    # print("Dados salvos com sucesso.")

    ## -----Essa forma substitui os IDs que forem iguais
    delta_table.alias("tabela_existente").merge(
        df_flattened.alias("novos_dados"),
        "tabela_existente._id = novos_dados._id AND tabela_existente.products_id = novos_dados.products_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print("Novos Produtos Adicionados com sucesso.")

else:
    print("Nenhum dado coletado. Nenhuma tabela será criada.")


StatementMeta(, df38d000-bf81-4fc1-8ef3-726ec4ecf875, 3, Finished, Available, Finished)

Total de páginas: 102
Processando página 1/102...
Processando página 2/102...
Processando página 3/102...
Processando página 4/102...
Processando página 5/102...
Processando página 6/102...
Processando página 7/102...
Processando página 8/102...
Processando página 9/102...
Processando página 10/102...
Processando página 11/102...
Processando página 12/102...
Processando página 13/102...
Processando página 14/102...
Processando página 15/102...
Processando página 16/102...
Processando página 17/102...
Processando página 18/102...
Processando página 19/102...
Processando página 20/102...
Processando página 21/102...
Processando página 22/102...
Processando página 23/102...
Processando página 24/102...
Processando página 25/102...
Processando página 26/102...
Processando página 27/102...
Processando página 28/102...
Processando página 29/102...
Processando página 30/102...
Processando página 31/102...
Processando página 32/102...
Processando página 33/102...
Processando página 34/102...
P